# Building RAG with Milvus and DeepSeek

DeepSeek helps developers build and scale AI applications using high-performance language models. It provides efficient inference, flexible APIs, and an advanced mixture-of-experts (MoE) architecture for powerful reasoning and retrieval tasks.

In this tutorial, we show how to build a retrieval-augmented generation (RAG) pipeline using Milvus and DeepSeek.

## Setup

### Dependencies and Environment

In [ ]:
!pip install openai==1.82.0 requests==2.32.3 tqdm==4.67.1 torch==2.7.0

---

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# Get the Local API key from environment variables
api_key = os.getenv("LOCAL_API_KEY")
print(f"Local API Key: {api_key}")
local_model ="qwen3.7-plus"

### Prepare Data

We use the Milvus documentation 2.4.x FAQ pages as the private knowledge base for our RAG pipeline; this is a good data source for a simple RAG example.

Download the zip file and extract the documents into the `milvus_docs` folder.

**It is recommended to run the following commands in the terminal**

In [ ]:
#!wget https://github.com/milvus-io/milvus-docs/releases/download/v2.4.6-preview/milvus_docs_2.4.x_en.zip
#!unzip -q milvus_docs_2.4.x_en.zip -d milvus_docs

We load all markdown files from the `milvus_docs/en/faq` folder. For each document, we split the content on "# " to roughly separate the main sections in the markdown files.

In [3]:
from glob import glob

text_lines = []

for file_path in glob("src/milvus_docs/en/faq/*.md", recursive=True):
    with open(file_path, "r") as file:
        file_text = file.read()

    text_lines += file_text.split("# ")

In [4]:
len(text_lines)

72

### Prepare the LLM and Embedding Model

DeepSeek supports the OpenAI-style API, so you can use a similar interface to call the LLM.

In [40]:
from openai import OpenAI

# deepseek_client = OpenAI(
#     api_key=api_key,
#     # base_url="https://api.deepseek.com/v1",  # DeepSeek API base URL
#     base_url="https://openrouter.ai/api/v1",
# )


local_client = OpenAI(
    base_url="http://192.168.40.64:8100/v1",
    api_key="LOCAL_API_KEY",
)
print(f"Local Client: {local_client}")

Local Client: <openai.OpenAI object at 0x000002CB09D02890>


Define an embedding model using `milvus_model` to generate text embeddings. We use `DefaultEmbeddingFunction`, a pretrained lightweight embedding model that runs fully locally without an API key.

In [18]:
# from pymilvus import model as milvus_model

# # Use Milvus built-in local embedding model, running fully offline
# embedding_model = milvus_model.DefaultEmbeddingFunction()
# from sentence_transformers import SentenceTransformer
from src.embedder import Embedder
from src.config import Config

config = Config()
embedding_model = Embedder()

# 1. Load a pretrained Sentence Transformer model
# embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = embedding_model.encode_queries(sentences)
print(embeddings.shape)
# [3, 384]

# 3. Calculate the embedding similarities
# similarities = embedding_model.similarity(embeddings, embeddings)
# print(similarities)
# # tensor([[1.0000, 0.6660, 0.1046],
# #         [0.6660, 1.0000, 0.1411],
# #         [0.1046, 0.1411, 1.0000]])


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6542.58it/s]


(3, 384)


Generate a test embedding and print its dimension and the first few values.

In [19]:
test_embedding = embedding_model.encode_queries(["This is a test"])[0]
embedding_dim = len(test_embedding)
print(embedding_dim)
print(test_embedding[:10])

384
[ 0.03061248  0.01383142 -0.02084379  0.0163279  -0.01023151 -0.0479843
 -0.01731333  0.03728748  0.04588735  0.034405  ]


## Load data into Milvus

len(text_lines)

### Prepare the LLM and Embedding Model

DeepSeek supports the OpenAI-style API, so you can use a similar interface to call the LLM.

In [ ]:
# from openai import OpenAI

# deepseek_client = OpenAI(
#     api_key=api_key,
#     # base_url="https://api.deepseek.com/v1",  # DeepSeek API base URL
#     base_url="https://openrouter.ai/api/v1",
# )

Define an embedding model using `milvus_model` to generate text embeddings. We use the `DefaultEmbeddingFunction` model as an example; it is a pretrained lightweight embedding model.

In [ ]:
# from pymilvus import model as milvus_model

# embedding_model = milvus_model.DefaultEmbeddingFunction()

from pymilvus import model as milvus_model

embedding_model = milvus_model.DefaultEmbeddingFunction()
# OpenAI domestic proxy https://api.apiyi.com/token 
# embedding_model = milvus_model.dense.OpenAIEmbeddingFunction(
#     model_name='nvidia/llama-nemotron-embed-vl-1b-v2:free', # Specify the model name
#     api_key=api_key, # Provide your OpenAI API key
#     base_url='https://openrouter.ai/api/v1',
#     dimensions=512
# )

Generate a test embedding and print its dimension and the first few values.

In [20]:
test_embedding = embedding_model.encode_queries(["This is a test"])[0]
embedding_dim = len(test_embedding)
print(embedding_dim)
print(test_embedding[:10])

384
[ 0.03061248  0.01383142 -0.02084379  0.0163279  -0.01023151 -0.0479843
 -0.01731333  0.03728748  0.04588735  0.034405  ]


In [21]:
test_embedding_0 = embedding_model.encode_queries(["That is a test"])[0]
print(test_embedding_0[:10])

[ 0.04992697  0.01530658 -0.01478047  0.01978594 -0.00713998 -0.03579961
  0.00823074  0.03300445  0.02352734  0.03255139]


## Load data into Milvus

### Create Collection

In [22]:
# from pymilvus import MilvusClient

# milvus_client = MilvusClient(uri="./milvus_demo.db")

# collection_name = "my_rag_collection"

from pymilvus import MilvusClient

milvus_client = MilvusClient(
    uri="http://localhost:19530"
)

collection_name = "my_rag_collection"
print(milvus_client.list_collections())

['my_rag_collection']


About `MilvusClient` parameters:

*   Setting `uri` to a local file such as `./milvus.db` is the easiest option because it uses Milvus Lite to store all data locally.
*   If you have larger data, you can use a higher-performance Milvus server running on Docker or Kubernetes. In that case, use the server URI such as `http://localhost:19530` as your `uri`.
*   If you want to use Zilliz Cloud (Milvus’s fully managed cloud service), adjust `uri` and `token` to match the Public Endpoint and API key from Zilliz Cloud.

Check whether the collection already exists, and drop it if it does.

In [23]:
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

Create a new collection with the specified parameters.

If we do not define any field schema, Milvus automatically creates a default `id` field as the primary key and a `vector` field to store vector data. A reserved JSON field is used to store fields and values that are not defined in the schema.

`metric_type` (distance metric):
     Purpose: defines how similarity between vectors is computed.
     Example: `IP` (inner product) - larger values are generally more similar; `L2` (Euclidean distance) - smaller values are more similar; `COSINE` (cosine similarity) - usually converted to distance, where smaller values are more similar.
     Choose based on your embedding model characteristics and your similarity definition.

`consistency_level` (consistency level):
     Purpose: defines how quickly reads see newly written data.
     Example:
         `Strong`: always reads the latest data, potentially slower.
         `Bounded`: may read slightly stale data for better performance (default).
         `Session`: reads your own writes immediately.
         `Eventually`: eventually reads new data without a strict guarantee, offering the best performance.
     Choose based on your requirements for data freshness versus performance.

In short:
 `metric_type`: how similarity is measured.
 `consistency_level`: how soon new data becomes visible.

In [24]:
milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim,
    metric_type="IP",  # inner product distance
    consistency_level="Strong",  # supported values are (`"Strong"`, `"Session"`, `"Bounded"`, `"Eventually"`). See https://milvus.io/docs/consistency.md#Consistency-Level for details.
)

### Insert Data

Iterate through the text lines, create embeddings, and insert the data into Milvus.

There is an extra field `text`, which is not defined in the collection schema. It will be automatically stored in Milvus’s reserved JSON dynamic field, which can be treated as a regular field at a high level.

In [25]:
from tqdm import tqdm

data = []

doc_embeddings = embedding_model.encode_documents(text_lines)

for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):
    data.append({"id": i, "vector": doc_embeddings[i], "text": line})

milvus_client.insert(collection_name=collection_name, data=data)

Creating embeddings: 100%|██████████| 72/72 [00:00<00:00, 35989.74it/s]


{'insert_count': 72, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71], 'cost': 0}

## Build RAG

### Retrieve query data

We specify a common question about Milvus.

In [26]:
question = "How is data stored in milvus?"

Search the collection for this question and retrieve the top 3 semantically matched results.

In [27]:

search_res = milvus_client.search(
    collection_name=collection_name,
    data=embedding_model.encode_queries(
        [question]
    ),  # convert the question to an embedding vector
    limit=3,  # return the top 3 results
    search_params={"metric_type": "IP", "params": {}},  # inner product distance
    output_fields=["text"],  # return the text field
)


Let's inspect the search results for the query.

In [28]:
import json

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))

[
    [
        " Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###",
        0.6488018035888672
    ],
    [
        "How does Milvus flush data?\n\nMilvus returns success when inserted data are loaded to t

### Use LLM to get RAG response

Convert the retrieved documents into a string format.

In [29]:
context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

In [ ]:
context

In [ ]:
question

Define system and user prompts for the language model. This prompt is assembled from documents retrieved from Milvus.

In [30]:
SYSTEM_PROMPT = """
Human: You are an AI assistant. You can find the answer to the question from the provided context paragraphs.
"""
USER_PROMPT = f"""
Please use the information fragments enclosed in <context> tags below to answer the question enclosed in <question> tags. Finally include the original answer's Chinese translation and mark it with <translated> and </translated> tags.
<context>
{context}
</context>
<question>
{question}
</question>
<translated>
</translated>
"""


In [ ]:
USER_PROMPT

Use the `deepseek-chat` model provided by DeepSeek to generate a response based on the prompts.

In [41]:
response = local_client.chat.completions.create(
    model= local_model,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)
print(response.choices[0].message.content)

# Answer to the Question

In Milvus, data is stored in two main ways:

1. **Inserted data** (vector data, scalar data, and collection-specific schema): These are stored in persistent storage as incremental logs. Milvus supports multiple object storage backends including MinIO, AWS S3, Google Cloud Storage (GCS), Azure Blob Storage, Alibaba Cloud OSS, and Tencent Cloud Object Storage (COS).

2. **Metadata**: Generated within Milvus and each Milvus module has its own metadata that are stored in etcd.

<translated>
在 Milvus 中，数据以两种主要方式存储：

1. 插入的数据（向量数据、标量数据和集合特定模式）：这些数据以增量日志的形式存储在持久化存储中。Milvus 支持多种对象存储后端，包括 MinIO、AWS S3、Google Cloud Storage (GCS)、Azure Blob Storage、Alibaba Cloud OSS 和 Tencent Cloud Object Storage (COS)。

2. 元数据：在 Milvus 内部生成，每个 Milvus 模块都有自己的元数据，存储在 etcd 中。
</translated>
